In [0]:
# ==============================================================================
# PIPELINE: 02_Ingest_Transaction_Silver
# ARCHITECTURE: Medallion Architecture (Bronze -> Silver Layer)
# DESIGN PATTERNS: foreachBatch, Quarantine Pattern, Ratio-Based Circuit Breaker,
#                  Idempotent Delta Writes
# ==============================================================================
#
# PURPOSE:
#   Incrementally reads new records from the Bronze table, validates them, and
#   splits each micro-batch into two destinations:
#     - Clean rows       -> realtime_fraud_detection.02_silver.cleaned_transactions
#     - Rejected rows    -> realtime_fraud_detection.02_silver.quarantine_transactions
#
# 1. foreachBatch Callback Pattern (.foreachBatch):
#    - Receives each micro-batch as a regular DataFrame, which allows multi-step
#      validation and writing to two Delta tables from a single batch.
#    - Runs with trigger(availableNow=True): processes all data available since
#      the last checkpoint, then stops. Suitable for scheduled, incremental runs.
#    - The checkpoint tracks which Bronze data has already been processed, so
#      reruns continue from where the previous run ended.
#
# 2. Single-Pass Validation with Rejection Reasons:
#    - Every row is tagged with a `_reason` in one pass. The first matching rule wins:
#        null_required_field : Time, Amount, or Class is NULL
#        invalid_amount      : Amount is negative or NaN
#        invalid_class       : Class is not 0 or 1
#        duplicate           : repeated row within the same micro-batch
#                              (the first occurrence is kept, extra copies are rejected)
#    - Rows with no reason are clean. All others are quarantined.
#    - Duplicates are quarantined rather than silently dropped, so nothing is
#      lost and every rejection can be audited.
#
# 3. Quarantine Pattern:
#    - Rejected rows are stored with extra audit columns:
#        _reason          : why the row was rejected
#        _batch_id        : the micro-batch (epoch) that produced it
#        _quarantined_at  : timestamp of quarantine
#    - Keeps Silver clean while preserving the bad data for debugging and replay.
#
# 4. Ratio-Based Circuit Breaker:
#    - Halts the job only if the share of rejected rows in a batch exceeds
#      MAX_BAD_RATIO (default 10%), which indicates systemic upstream corruption.
#    - A ratio is used instead of a fixed row count so that large backfills do
#      not trigger false alarms, while real corruption spikes are still caught.
#    - The check runs BEFORE any write, so a tripped breaker leaves both Silver
#      and Quarantine untouched for that batch.
#
# 5. Fault Tolerance and Idempotency:
#    - foreachBatch is at-least-once: a failed batch can be retried.
#    - Delta options txnAppId + txnVersion (= epoch_id) make each write
#      idempotent, so a retry will not insert the same batch twice.
#    - Each table uses its own txnAppId, so the two writes are tracked separately.
#    - NOTE: if the checkpoint is ever reset, batch IDs restart from 0 and writes
#      would be skipped. Change the txnAppId values when resetting a checkpoint.
#    - Cached DataFrames are released in a finally block, even when the circuit
#      breaker raises an error.
#
# KNOWN LIMITATIONS:
#   - Deduplication works only WITHIN a single micro-batch. The same record
#     arriving in a different batch is not detected. Use a MERGE into Silver if
#     cross-batch deduplication is required.
#   - DEDUP_COLS = None deduplicates on all columns. Set it to your true business
#     key (or a transaction ID) for production use.
#   - Verify schema names (`01_bronze`, `02_silver`) and column casing
#     (Time, Amount, Class) against your actual tables before the first run.
# ==============================================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_TABLE     = "realtime_fraud_detection.`01_bronze`.raw_transactions"  # verify names
SILVER_TABLE     = "realtime_fraud_detection.`02_silver`.cleaned_transactions"
QUARANTINE_TABLE = "realtime_fraud_detection.`02_silver`.quarantine_transactions"
CHECKPOINT       = "/Volumes/realtime_fraud_detection/00_landing/raw_data/transactions/_checkpoints/"

MAX_BAD_RATIO = 0.10
DEDUP_COLS = None  # set to your business columns; None = all columns

def process_micro_batch(batch_df, epoch_id):
    batch_df.persist()
    try:
        dedup_cols = DEDUP_COLS or batch_df.columns
        w = Window.partitionBy(*dedup_cols).orderBy(F.lit(1))

        flagged = (
            batch_df
            .withColumn("_rn", F.row_number().over(w))
            .withColumn(
                "_reason",
                F.when(F.col("Time").isNull() | F.col("Amount").isNull() | F.col("Class").isNull(),
                       "null_required_field")
                 .when((F.col("Amount") < 0) | F.isnan("Amount"), "invalid_amount")
                 .when(~F.col("Class").isin(0, 1), "invalid_class")
                 .when(F.col("_rn") > 1, "duplicate")
            )
        ).persist()

        total = flagged.count()
        if total == 0:
            return

        clean = flagged.filter(F.col("_reason").isNull()).drop("_rn", "_reason")
        bad = (flagged.filter(F.col("_reason").isNotNull())
               .drop("_rn")
               .withColumn("_batch_id", F.lit(epoch_id))
               .withColumn("_quarantined_at", F.current_timestamp()))

        bad_count = bad.count()
        bad_ratio = bad_count / total
        print(f"Batch {epoch_id}: {total} rows, {bad_count} quarantined ({bad_ratio:.2%})")

        if bad_ratio > MAX_BAD_RATIO:
            raise ValueError(f"Batch {epoch_id}: bad ratio {bad_ratio:.2%} exceeds {MAX_BAD_RATIO:.0%}")

        (clean.write.format("delta").mode("append")
              .option("txnAppId", "silver_clean").option("txnVersion", epoch_id)
              .saveAsTable(SILVER_TABLE))

        if bad_count > 0:
            (bad.write.format("delta").mode("append")
                .option("txnAppId", "silver_quarantine").option("txnVersion", epoch_id)
                .saveAsTable(QUARANTINE_TABLE))
    finally:
        batch_df.unpersist()
        try:
            flagged.unpersist()
        except NameError:
            pass

query = (
    spark.readStream.table(BRONZE_TABLE)
    .writeStream
    .foreachBatch(process_micro_batch)
    .trigger(availableNow=True)
    .option("checkpointLocation", CHECKPOINT)
    .start()
)
query.awaitTermination()
print("Silver ingestion finished.")